-------------------------------------
# **Estudo de Caso - Análise de Dados da Uber**
-------------------------------------

--------------------
## **Contexto**
--------------------

A Uber conecta passageiros e motoristas por meio de um aplicativo móvel e fornece milhões de corridas todos os dias.

A demanda por corridas muda dependendo do horário, localização, clima e eventos. Para ter sucesso, a Uber precisa entender e prever essas mudanças.

Você é um Cientista de Dados no escritório da Uber em Nova York. Sua tarefa é analisar os dados, encontrar padrões e sugerir ações que possam ajudar a empresa a tomar melhores decisões.

------------------
## **Objetivo**
------------------

O objetivo é aprender a explorar dados e extrair insights simples.

------------------------------------
## **Descrição do Conjunto de Dados**
------------------------------------

Os dados contêm informações sobre clima, localização e número de embarques (pickups).

* pickup_dt: Data e hora do embarque
* borough: Município de Nova York (borough)
* pickups: Número de embarques para o período (1 hora)
* spd: Velocidade do vento em milhas/hora
* vsb: Visibilidade em milhas (arredondado para o décimo mais próximo)
* temp: Temperatura em Fahrenheit
* dewp: Ponto de orvalho em Fahrenheit
* slp: Pressão ao nível do mar
* pcp01: Precipitação líquida em 1 hora
* pcp06: Precipitação líquida em 6 horas
* pcp24: Precipitação líquida em 24 horas
* sd: Profundidade da neve em polegadas
* hday: Indica se é feriado (Y) ou não (N)

## **Fase 1: Limpeza dos dados (Data Cleaning)**

### Importando as bibliotecas necessárias e visão geral do dataset

In [ ]:
# Biblioteca para suprimir avisos
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Bibliotecas para ajudar na leitura e manipulação de dados
import pandas as pd
import matplotlib.pyplot as plt

# Bibliotecas para ajudar na visualização de dados
import seaborn as sns

### Carregando o conjunto de dados

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
data = pd.read_csv('/content/drive/MyDrive/pucsp-deeplearning/I2.1-Uber_DataSet.csv')


In [ ]:
# Copiando os dados para outra variável para evitar alterações nos dados originais
df = data.copy()

### Visualizando as 5 primeiras linhas do conjunto de dados

In [ ]:
# Visualizando o head (as 5 primeiras observações)
df.head()

**Observações:**

* A coluna pickup_dt inclui a data e hora do embarque. A data mostra que os dados começam em 01-Jan-2015.
* A coluna borough contém o nome do município de Nova York em que o embarque foi realizado.
* A coluna pickups contém o número de embarques no município no horário determinado.
* Todas as variáveis climáticas são numéricas.
* A variável hday (feriado) é uma variável categórica.

### Verificando a forma (shape) do conjunto de dados

In [ ]:
df.shape

* O conjunto de dados possui **29.101 linhas e 13 colunas**.

### Verificando o info()

In [ ]:
df.info()

**Observações:**

* Todas as colunas têm 29.101 observações, exceto borough, que tem 26.058 observações, indicando que há valores nulos nela.
* pickup_dt é lida como um tipo de dado 'object', mas deveria ser do tipo DateTime.
* borough e hday (feriado) deveriam ser variáveis categóricas.

### Resumo estatístico dos dados

In [ ]:
df.describe().T

* Há uma discrepância significativa entre o terceiro quartil e o valor máximo para o número de embarques (pickups) e a profundidade da neve (sd), indicando que essas variáveis podem ter outliers à direita.
* A temperatura tem uma ampla faixa de variação, mostrando que os dados incluem registros tanto do inverno quanto do verão.

### Tratamento de valores ausentes

In [ ]:
# Verificando valores ausentes
df.isna().sum()

* Há 3043 valores ausentes para a variável borough.
* As outras variáveis não possuem valores ausentes.

In [ ]:
# Substituindo NaN por "Unknown" (Desconhecido)
df['borough'].fillna('Unknown', inplace = True)

In [ ]:
df.borough.value_counts()

In [ ]:
df.isnull().sum()

* Agora, não há mais valores ausentes nos dados.

## **Fase 2: Engenharia de Atributos (Feature Engineering)**

### Extraindo partes da data de embarque

In [ ]:
# Convertendo o tipo de dado de pickup_dt para datetime
df.pickup_dt = pd.to_datetime(df.pickup_dt)

# Extraindo partes da data de pickup_dt
df['start_year'] = df.pickup_dt.dt.year

df['start_month'] = df.pickup_dt.dt.month_name()

df['start_hour'] = df.pickup_dt.dt.hour

df['start_day'] = df.pickup_dt.dt.day

df['week_day'] = df.pickup_dt.dt.day_name()

In [ ]:
# Removendo a coluna pickup_dt, pois não será mais necessária para análises posteriores
df.drop('pickup_dt', axis = 1, inplace = True)

In [ ]:
df.info()

## **Fase 3: Análise Exploratória (EDA) e Insights**

### **Relação entre o número de embarques e variáveis baseadas em tempo**

### Embarques por Mês

In [ ]:
cats = df.start_month.unique().tolist()
df.start_month = pd.Categorical(df.start_month, ordered = True, categories = cats)
plt.figure(figsize = (20, 7))
sns.lineplot(x = "start_month", y = "pickups", data = df, ci = 0, color = "RED", estimator = 'sum')
plt.ylabel('Total de embarques')
plt.xlabel('Mês')
plt.show()

**Observações:**
* Há uma clara tendência de aumento nos embarques mensais.
* Os embarques em junho são quase 1,5 vezes maiores do que em janeiro.

### Embarques por Dia do Mês

In [ ]:
plt.figure(figsize = (20, 7))
sns.lineplot(x = "start_day", y = "pickups", estimator = 'sum', ci = 0, data = df, color = "RED")
plt.ylabel('Total de embarques')
plt.xlabel('Dia do Mês')
plt.show()

**Observações:**
* O número de embarques é baixo no final do mês (dias 29 a 31).
* O número de embarques no dia 31 pode ser baixo porque nem todos os meses têm 31 dias.
* Há um pico nos embarques por volta do dia 20 do mês.

### Embarques por Dia da Semana

In [ ]:
cats = ['Monday', 'Tuesday', 'Wednesday','Thursday', 'Friday', 'Saturday', 'Sunday']
df.week_day = pd.Categorical(df.week_day, ordered = True, categories = cats)
plt.figure(figsize = (20, 7))
sns.lineplot(x = "week_day", y = "pickups", ci = 0, data = df, color = "RED")
plt.ylabel('Média de embarques')
plt.xlabel('Dia da Semana')
plt.show()

**Observações:**
* Os embarques aumentam gradualmente conforme a semana avança e começam a cair depois de sábado.
* Precisamos investigar melhor para entender por que a demanda por Uber é baixa no início da semana.

### Embarques por Município (Borough)

In [ ]:
plt.figure(figsize = (20, 10))
sns.boxplot(x='borough', y='pickups', data=df)
plt.ylabel('Embarques')
plt.xlabel('Município (Borough)')
plt.show()

**Observações:**
* Há uma diferença clara no número de passageiros entre os diferentes municípios.
* Manhattan tem o maior número de corridas.
* Brooklyn e Queens são seguidores distantes.
* EWR (Newark), Unknown (Desconhecido) e Staten Island têm um número muito baixo de corridas. A demanda é tão pequena que provavelmente pode ser coberta pelos desembarques das viagens de entrada de outras áreas.

## **Conclusão e Recomendações**

-----------------------------------------------------------------
### **Conclusão**
-----------------------------------------------------------------

Analisamos cerca de 30.000 registros de embarques horários da Uber em Nova York durante os primeiros seis meses de 2015. Nosso foco principal é o número de corridas. Do ponto de vista comercial, é ineficiente ter motoristas no lugar errado ou na hora errada, por isso queremos entender quais fatores influenciam a demanda.

Conseguimos concluir que:

1. Os táxis da Uber são mais populares na área de Manhattan, em Nova York.
2. A demanda por Uber tem aumentado constantemente ao longo dos meses (janeiro a junho).
3. A taxa de embarques é maior nos fins de semana em comparação com os dias de semana.
4. É encorajador ver que os nova-iorquinos confiam nos serviços de táxi da Uber quando saem para aproveitar suas noites.
5. Precisamos investigar mais a fundo a baixa demanda por Uber nas segundas-feiras.

--------------------------------------------------
### **Recomendações para o negócio**
--------------------------------------------------

1. Manhattan é o mercado mais maduro para a Uber. Brooklyn, Queens e Bronx mostram potencial.
2. Houve um aumento gradual nas corridas da Uber nos últimos meses, e precisamos manter o ritmo.
3. O número de passageiros é alto nos horários de pico do expediente durante a semana e nas noites de sábado. A disponibilidade de carros deve ser garantida nesses horários.
4. A demanda por carros é maior nas noites de sábado. A disponibilidade de carros deve ser garantida nesse dia da semana.
5. Obter dados sobre o tamanho da frota disponível para ter uma melhor compreensão da situação de oferta e demanda e construir um modelo de machine learning para prever com precisão os embarques por hora, a fim de otimizar a frota de carros nas respectivas áreas.